## df = spark.read.format("delta").table("schema.tabela")

Gerando um dataframe dos delta lake no container bronze do Azure Data Lake Storage

In [0]:
df_credits   = spark.read.format("delta").table("silver_filmes.credits")
df_keywords  = spark.read.format("delta").table("silver_filmes.keywords")
df_links   = spark.read.format("delta").table("silver_filmes.links")
df_links_small = spark.read.format("delta").table("silver_filmes.links_small")
df_movies_metadata = spark.read.format("delta").table("silver_filmes.movies_metadata")
df_ratings   = spark.read.format("delta").table("silver_filmes.ratings")
df_ratings_small = spark.read.format("delta").table("silver_filmes.ratings_small")

### Adicionando metadados de data e hora de processamento e nome do arquivo de origem

In [0]:
%sql
drop table if exists gold_filmes.dim_links

In [0]:
%sql
CREATE TABLE gold_filmes.dim_links (
  sk_links bigint generated by default as identity,
  movie_id BIGINT,
  imdb_id BIGINT,
  tmdb_id BIGINT
)
USING DELTA;


In [0]:
%sql
DESCRIBE TABLE EXTENDED gold_filmes.dim_links


col_name,data_type,comment
sk_links,bigint,null
movie_id,bigint,null
imdb_id,bigint,null
tmdb_id,bigint,null
,,
# Detailed Table Information,,
Catalog,workspace,
Database,gold_filmes,
Table,dim_links,
Created Time,Tue Oct 14 02:06:26 UTC 2025,


In [0]:
df_links.createOrReplaceTempView("links")
df_links_small.createOrReplaceTempView("links_small")

In [0]:
%sql
WITH links_relacional AS (
  SELECT
    movie_id,
    imdb_id,
    tmdb_id
  FROM links_small
)

MERGE INTO gold_filmes.dim_links AS l
USING links_relacional AS lr
ON l.movie_id = lr.movie_id

WHEN MATCHED THEN
  UPDATE SET
    l.imdb_id = lr.imdb_id,
    l.tmdb_id = lr.tmdb_id

WHEN NOT MATCHED THEN
  INSERT (movie_id, imdb_id, tmdb_id)
  VALUES (lr.movie_id, lr.imdb_id, lr.tmdb_id)


num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
9125,0,0,9125


In [0]:
%sql
select * from gold_filmes.dim_links

sk_links,movie_id,imdb_id,tmdb_id
1,1,114709,862
2,2,113497,8844
3,3,113228,15602
4,4,114885,31357
5,5,113041,11862
6,6,113277,949
7,7,114319,11860
8,8,112302,45325
9,9,114576,9091
10,10,113189,710
